In [2]:
import pandas as pd
import numpy as np

# Todo:
# clean authors to get all locations for each author
# clean topics, sub field, field, domain same as authors
# change get scripts to include author name


In [94]:
# cleaning authors
# problem: some authors don't have all their locations listed for each work. We need to find all unique locations for each author and then add that full list to each author entry in the authors table
# The locations of each author are kept in a list under the column name "countries"
# this will be run on lots of data, so needs to be efficient and use primarily pandas or numpy operations.
from ast import literal_eval

authors_df = pd.read_csv('./authors.csv')
# authors_df.loc[authors_df["countries"].eq('[]'), "countries"] = ''
# authors_df['countries_all'] = authors_df['countries'].apply(literal_eval)  # convert string representation of list to actual list

authors_df['countries'] = authors_df['countries'].str[1:-1]
authors_df = authors_df.groupby('a_id')['countries'].agg(list).apply(lambda x: list(set(x))).reset_index()
# authors_df = (authors_df.groupby('a_id', as_index=False)['countries']
#               .agg(lambda x: list(x.dropna().unique())))
authors_df["countries_str"] = authors_df["countries"].astype(str)
# authors_df['countries_all'] = authors_df['countries'].apply(lambda x: list(set(x)))
# authors_df
authors_df[authors_df['a_id'] == "https://openalex.org/A5000539614"]

# merge the locations back to the authors_df
# authors_df = authors_df.merge(author_locations.rename('all_locations'), left_on='a_id', right_index=True, how='left')

  



,a_id,countries,countries_str
789,https://openalex.org/A5000539614,"[, 'US']","['', ""'US'""]"


In [43]:
#test the function
# use authors csv from data/processed/authors.csv
authors_df = pd.read_csv('./authors.csv')
cleaned_authors_df = clean_authors(authors_df)
# If an author has no locations (just a [[]]), fill with NaN
cleaned_authors_df['all_locations'] = cleaned_authors_df['all_locations'].apply(lambda x: x if x != [[]] else np.nan)

# cleaned_authors_df.head(100)
cleaned_authors_df[authors_df['a_id'] == "https://openalex.org/A5000539614"]

KeyError: 'countries'

In [ ]:
# data tables: 
# topic_field_display_name, num_publications, publication_year
# a_id, publication_order, topic_field_display_name (if previous work was their last, then this is "No NUMBER publication"), topic_field_display_name of first publication
# year, num_publications, num_authors, topic_field_display_name, cross-colaberation metric
# metric_name (total_publications, total_authors, total citation count, total distinct keywords), value
#

In [161]:
import pandas as pd

works_df = pd.read_csv('./works_sampled_large.csv')
topics_df = pd.read_csv('./topics_sampled_large.csv')

works_df = works_df.merge(topics_df[['work_id', 'topic_field_display_name']], left_on='id', right_on='work_id', how='inner')

works_df = works_df.groupby(["pub_year", "topic_field_display_name"])["id"].count().reset_index()

works_df.sort_values(by=["id"], inplace=True)

# sum all id values prior to 1900
works_df_pre_1900 = works_df[works_df["pub_year"] < 1900].groupby("topic_field_display_name")["id"].sum().reset_index()
works_df = works_df[works_df["pub_year"] >= 1900]

all_works_df = pd.concat([works_df_pre_1900.assign(pub_year=1899), works_df], ignore_index=True)

all_works_df.to_csv('./topic_field_publications_per_year.csv', index=False)

,topic_field_display_name,id,pub_year
0,Agricultural and Biological Sciences,6370,1899
1,Arts and Humanities,17787,1899
2,"Biochemistry, Genetics and Molecular Biology",3399,1899
3,"Business, Management and Accounting",1013,1899
4,Chemical Engineering,258,1899
...,...,...,...
3297,Medicine,70288,2023
3298,Social Sciences,73056,2023
3299,Social Sciences,73062,2021
3300,Social Sciences,81836,2018


In [177]:
# # a_id, publication_order, topic_field_display_name (if previous work was their last, then this is "No NUMBER publication"), topic_field_display_name of first publication

import pandas as pd

topics_df = pd.read_csv('./topics_sampled_large.csv')
# get only topics that are the primary topic of the work
topics_df = topics_df.sort_values('topic_score').groupby(['work_id']).last().reset_index()
# topics_df[topics_df['work_id'] == 'https://openalex.org/W2460813123']
authors_df = pd.read_csv('./authors_sampled_large.csv')
authors_df = authors_df[authors_df['a_id'] != "https://openalex.org/A9999999999"]
works_df = pd.read_csv('./works_sampled_large.csv')


authors_df = authors_df.merge(works_df[['id', 'pub_date']], left_on='work_id', right_on='id', how='inner')
authors_df = authors_df[['a_id', 'work_id', 'pub_date']].merge(topics_df[['work_id', 'topic_field_display_name', 'topic_score']], left_on='work_id', right_on='work_id', how='inner')
authors_df.sort_values(by=['a_id', 'work_id'], inplace=True)

authors_df['publication_order'] = authors_df.groupby(['a_id'])['pub_date'].rank(method= 'first' ,ascending=True).astype(int)
authors_df.sort_values(by=['a_id', 'publication_order'], inplace=True)
authors_df['first_publication_topic'] = authors_df.groupby('a_id')['topic_field_display_name'].transform('first')
# add an extra publication and topic_field_display_name that says "No Next Publication" and has a publication_order of max + 1 for each author

# max_publication_order = authors_df.groupby(['a_id','first_publication_topic'])['publication_order'].max().reset_index()
# max_publication_order['publication_order'] += 1
# max_publication_order['topic_field_display_name'] = "No Next Publication"
# # max_publication_order.sort_values(by=['a_id'])
# authors_df = pd.concat([authors_df, max_publication_order], ignore_index=True, sort=False)
authors_df = authors_df[['a_id', 'publication_order', 'topic_field_display_name', 'first_publication_topic']]
authors_df.sort_values(by=['a_id', 'publication_order'], inplace=True)
# authors_df.to_csv('./author_publication_topics_sankey.csv', index=False)

# assign a number to each unique topic_field_display_name for each author based on publication_order
authors_df_topic_ranks = authors_df.groupby(['a_id', 'topic_field_display_name'])['publication_order'].min().reset_index()
authors_df_topic_ranks.rename(columns={"publication_order": "topic_order"}, inplace=True)
authors_df_topic_ranks['topic_order'] = authors_df_topic_ranks.groupby(['a_id'])['topic_order'].rank(method= 'first' ,ascending=True).astype(int)


#merge back to authors_df
authors_df = authors_df.merge(authors_df_topic_ranks, on=['a_id', 'topic_field_display_name'], how='left')
authors_df = authors_df.groupby(['first_publication_topic','publication_order','topic_order']).count().reset_index()
authors_df.rename(columns={"a_id": "author_count"}, inplace=True)

authors_df = authors_df[['first_publication_topic','publication_order','topic_order','author_count']]
authors_df.to_csv('./author_publication_sankey.csv', index=False)
# authors_df[authors_df['a_id'] == "https://openalex.org/A5002382626"]
# authors_df

In [178]:
test = pd.read_csv('./author_publication_sankey.csv')
test

,first_publication_topic,publication_order,topic_order,author_count
0,Agricultural and Biological Sciences,1,1,326272
1,Agricultural and Biological Sciences,2,1,26392
2,Agricultural and Biological Sciences,2,2,37284
3,Agricultural and Biological Sciences,3,1,8655
4,Agricultural and Biological Sciences,3,2,7991
...,...,...,...,...
34029,Veterinary,542,2,1
34030,Veterinary,543,2,1
34031,Veterinary,544,2,1
34032,Veterinary,545,4,1


In [179]:
# year, num_publications, num_authors, topic_field_display_name, cross-colaberation metric
import pandas as pd
import gc
works_df = pd.read_csv('./works_sampled_large.csv')
works_df = works_df[works_df['pub_year']>= 1900]
topics_df = pd.read_csv('./topics_sampled_large.csv')

authors_df = pd.read_csv('./authors_sampled_large.csv')

# get most common topic_field_display_name for each author
authors_primary_topic_df = authors_df.merge(topics_df[['work_id', 'topic_field_display_name']], left_on='work_id', right_on='work_id', how='inner')
# get most common topic_field_display_name for each author
authors_primary_topic_df = authors_primary_topic_df.groupby(['a_id', 'topic_field_display_name']).size().reset_index(name='counts')
authors_primary_topic_df = authors_primary_topic_df.sort_values('counts', ascending=False).groupby('a_id').first().reset_index()
authors_primary_topic_df.rename(columns={"topic_field_display_name": "author_primary_topic"}, inplace=True)

authors_df = authors_df[['a_id', 'work_id']].merge(authors_primary_topic_df[['a_id', 'author_primary_topic']], left_on='a_id', right_on='a_id', how='inner')

topics_df = topics_df.sort_values('topic_score').groupby(['work_id']).last().reset_index()
# works_df
works_df = works_df[['id', 'pub_year']].merge(topics_df[['work_id', 'topic_field_display_name']], left_on='id', right_on='work_id', how='inner')

works_df = works_df.merge(authors_df[['a_id', 'work_id', 'author_primary_topic']], left_on='id', right_on='work_id', how='inner')

works_df = works_df[['id','pub_year', 'topic_field_display_name', 'a_id', 'author_primary_topic']]

works_df["topic_match"] = works_df.apply(lambda row: 0 if row['topic_field_display_name'] == row['author_primary_topic'] else 1, axis=1)
works_df.sort_values(by=['author_primary_topic', 'pub_year'], inplace=True)
works_df["author_cumsum"] = works_df.drop_duplicates(subset=['author_primary_topic','a_id']).groupby(['author_primary_topic']).cumcount()+1

works_df["author_cumsum"] = works_df.groupby(['author_primary_topic','pub_year'])['author_cumsum'].transform('max').fillna(0).astype(int)
# set author_cumsum to 0 where topic_field_display_name != author_primary_topic
works_df.loc[works_df['topic_field_display_name'] != works_df['author_primary_topic'], 'author_cumsum'] = 0

works_df = works_df.groupby(['pub_year', 'topic_field_display_name']).agg(
    num_publications=('id', 'nunique'),
    cross_collaboration_metric=('topic_match', 'mean'),
    authors = ('author_cumsum', 'max')
).reset_index()

# pd.set_option('display.max_rows', 100)

# works_df[works_df['topic_field_display_name'] == "Computer Science"].sort_values(by=['pub_year'], ascending= False).head(100)
works_df.to_csv('./data_for_bubble_chart_large.csv', index=False)

In [170]:
test = pd.read_csv('./data_for_bubble_chart_large.csv')
test

,pub_year,topic_field_display_name,num_publications,cross_collaboration_metric,authors
0,1900,Agricultural and Biological Sciences,71,0.808989,93
1,1900,Arts and Humanities,205,0.784906,277
2,1900,"Biochemistry, Genetics and Molecular Biology",36,0.680000,51
3,1900,"Business, Management and Accounting",19,0.428571,14
4,1900,Chemical Engineering,8,0.363636,5
...,...,...,...,...,...
3268,2025,"Pharmacology, Toxicology and Pharmaceutics",95,0.270103,12632
3269,2025,Physics and Astronomy,501,0.537274,167286
3270,2025,Psychology,878,0.489629,94954
3271,2025,Social Sciences,3935,0.722490,639834
